# 11 Communities and resistomes: vegan's dune, varespec, BCI and mite

simplydata's words for tables of counts of many kinds (species in plots, genes in samples), checked on the
standard datasets of community ecology that ship with R vegan:

- **BCI** (Condit et al. 2002): trees of 225 species in 50 one-hectare plots of Barro Colorado Island;
- **dune** and **dune.env** (Jongman, ter Braak & van Tongeren 1987): 30 plant species in 20 meadows, with the
  management of each;
- **varespec** and **varechem** (Väre, Ohtonen & Oksanen 1995): cover of 44 lichens and plants and 14 soil
  variables at 24 sites;
- **mite** and **mite.env** (Borcard & Legendre 1994): 35 oribatid mites in 70 soil cores, with the microtopography.

Each answer is compared with
- **published values** where they are printed: the help pages of vegan (`diversity` and `rarefy` on BCI) and
  Oksanen's vegan tutorial (correspondence analysis of varespec: eigenvalues 0.5249 and 0.3568 of 2.083), within
  half a unit of the last printed digit;
- **an independent calculation in Python** from the textbook formula (SciPy `entropy`, `pdist`, `linkage`,
  NumPy), where one exists: Shannon and Simpson, rarefaction (Hurlbert 1971), Bray-Curtis, PERMANOVA's pseudo-F
  and R squared (Anderson 2001), ANOSIM's R (Clarke 1993), the Mantel correlation, principal coordinates (Gower
  1966) and the log ratios;
- **R** (vegan, zCompositions, compositions, cluster), run here with the same calls.

**Only simplydata runs in this notebook.** The reference values (`references/11_communities.csv`, from
`scripts/references/11_communities.py`): the printed values, the formulas written out and R, each check with its
source. The NMDS stress and the correspondence-analysis shares are read with `show="numbers"`, and the joins of the
cluster tree with `chart.get_numbers_drawn()`. `show="numbers"` also gives back the correspondence-analysis
eigenvalues and the total inertia (simplydata a5a8ff0 and later). Whole tables (the 190 distances, the 70 x 35 log ratios) are compared by their largest
relative difference.

**Permutation p-values.** simplydata runs vegan's permutation tests in R after `set.seed(with_seed)` (default 1).
The R reference here runs the same steps in the same order after `set.seed(1)`, so the permutations are the same
and the p-values must agree exactly. Where a p-value could not be reproduced that way, it would be compared with
a Monte Carlo tolerance of 4 binomial standard errors, 4 sqrt(p (1 - p) / 999), and that is said where it is used.

Tolerances (the same for every dataset): 1e-12 relative for closed formulas (diversity, distances), 1e-10 for
results of an eigen- or least-squares decomposition (PERMANOVA, ordinations, constrained ordination), 1e-6 for
the iterative NMDS stress and for the log ratios after the iterative zero replacement (as README: models fitted
by iteration); permutation p-values with the same seed: exact (1e-12). An axis of an ordination has an arbitrary
sign, so ordination scores are compared by their absolute value.

In [1]:
import simplydata as sdata

from checks import ReferenceChecks, quiet, read_row, read_rows, versions

NOTEBOOK = "11_communities"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)


def load(name):
    """A fresh copy of a vegan dataset written to data/fetched/ by scripts/get_data.py (the add_ words add columns to the table they run on)."""
    return quiet(sdata.table, f"../data/fetched/{name}.csv", separator=",")


def rows(word, *args, **options):
    return read_rows(quiet(word, *args, **options))


DUNE_ENV = ["plot", "A1", "Moisture", "Management", "Use", "Manure"]
dune_species = [c for c in load("dune").get_column_names() if c not in DUNE_ENV]
VARECHEM = ["N", "P", "K", "Ca", "Mg", "S", "Al", "Fe", "Mn", "Zn", "Mo", "Baresoil", "Humdepth", "pH"]
vare_species = [c for c in load("varespec").get_column_names() if c not in VARECHEM + ["site"]]
mite_species = [c for c in load("mite").get_column_names() if c != "Topo"]

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit a5a8ff0 of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## Diversity of the 50 BCI plots: `add_diversity_columns`

Richness, Shannon (natural log) and Simpson (1 - sum of p²) of every plot, against vegan `specnumber` and
`diversity`, and against the formulas (SciPy `entropy`; NumPy). vegan's help page for `diversity` prints the
first three plots: Shannon 4.018, 3.848, 3.814 and Simpson 0.975, 0.968, 0.965 (`diversity(BCI)`). Also
`at_total_count=20` (expected richness in 20 trees; vegan `rarefy`, whose help page prints 16.361, 15.670 and
15.301 for the first three plots, and Hurlbert's formula), `show="chao1"` (vegan `estimateR`) and
`show="hill numbers"` (vegan `renyi(hill = TRUE)`; Hill 1 = exp(Shannon), Hill 2 = 1 / sum of p²).

In [2]:
species = list(load("bci").get_column_names())
first_three = load("bci")
quiet(first_three.keep_first_rows, 3)
first_three.add_diversity_columns(columns=species)

Diversity of 3 rows from 225 columns (Abarema.macradenia, Vachellia.melanoceras, Acalypha.diversifolia, Acalypha.macrostachya, Adelia.triloba, Aegiphila.panamensis and 219 more):
  Richness (the number of kinds present) averages 89.0 (from 84 to 93).
  Shannon diversity averages 3.89 (natural log): as diverse as a sample of 49.3 equally common kinds, on average (its effective number, Hill number q = 1).
  Simpson's index averages 0.97: the chance that two counts drawn at random are of different kinds (1 - the sum of squared shares); as diverse as 33.1 equally common kinds (inverse Simpson, Hill number q = 2).
Method: R vegan::specnumber(x), diversity(x, "shannon"), diversity(x, "simpson"), renyi(x, scales = c(0, 1, 2), hill = TRUE). simplydata chooses the natural log for Shannon, Simpson as 1 - D (the Gini-Simpson index), Chao1 bias-corrected and rarefaction as an expected value. Added richness, shannon and simpson.


,Abarema.macradenia,Vachellia.melanoceras,Acalypha.diversifolia,Acalypha.macrostachya,Adelia.triloba,Aegiphila.panamensis,Alchornea.costaricensis,Alchornea.latifolia,Alibertia.edulis,Allophylus.psilospermus,...,Vochysia.ferruginea,Xylopia.macrantha,Zanthoxylum.ekmanii,Zanthoxylum.juniperinum,Zanthoxylum.panamense,Zanthoxylum.setulosum,Zuelania.guidonia,richness,shannon,simpson
1,0,0,0,0,0,0,2,0,0,0,...,0,1,3,0,2,0,0,93.0,4.018412,0.974629
2,0,0,0,0,0,0,1,0,0,0,...,0,0,4,0,2,0,0,84.0,3.848471,0.968339
3,0,0,0,0,0,0,2,0,0,0,...,0,0,8,1,2,0,0,90.0,3.814060,0.964608


In [3]:
found = rows(load("bci").add_diversity_columns, columns=species)
for i, row in enumerate(found):
    checks.check_values(f"BCI plot {i + 1}", "add_diversity_columns",
                        {column: row[column] for column in ("richness", "shannon", "simpson")})

found = rows(load("bci").add_diversity_columns, columns=species, show="richness", at_total_count=20)
for i, row in enumerate(found):
    checks.check(f"BCI plot {i + 1}", "add_diversity_columns (at_total_count=20)", "expected richness",
                 row["expected richness at 20"])

chao = rows(load("bci").add_diversity_columns, columns=species, show="chao1")
hill = rows(load("bci").add_diversity_columns, columns=species, show="hill numbers")
for i in range(len(chao)):
    checks.check_values(f"BCI plot {i + 1}", "add_diversity_columns (chao1)",
                        {"chao1": chao[i]["chao1"], "chao1 standard error": chao[i]["chao1 standard error"]})
    checks.check_values(f"BCI plot {i + 1}", "add_diversity_columns (hill numbers)",
                        {"hill number 1": hill[i]["hill number 1"], "hill number 2": hill[i]["hill number 2"]})

agrees  BCI plot 1 | add_diversity_columns | richness: simplydata 93, formula (SciPy entropy, NumPy) 93; relative difference 0.00e+00 (allowed 1e-12)
      Source: formula (SciPy entropy, NumPy) (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.entropy.html
agrees  BCI plot 1 | add_diversity_columns | richness: simplydata 93, R vegan 93; relative difference 0.00e+00 (allowed 1e-12)
      Source: R vegan (run reference) - https://cran.r-project.org/package=vegan (version 2.7.8)
agrees  BCI plot 1 | add_diversity_columns | shannon: simplydata 4.01841166223, formula (SciPy entropy, NumPy) 4.01841166223; relative difference 8.84e-16 (allowed 1e-12)
      Source: formula (SciPy entropy, NumPy) (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.entropy.html
agrees  BCI plot 1 | add_diversity_columns | shannon: simplydata 4.01841166223, R vegan 4.01841166223; relative difference 0.00e+00 (allowed 1e-12)
      Sourc

agrees  BCI plot 1 | add_diversity_columns (chao1) | chao1: simplydata 117.473684211, R vegan::estimateR 117.473684211; relative difference 0.00e+00 (allowed 1e-12)
      Source: R vegan::estimateR (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/estimateR.html (version 2.7.8)
agrees  BCI plot 1 | add_diversity_columns (chao1) | chao1 standard error: simplydata 11.5837848487, R vegan::estimateR 11.5837848487; relative difference 0.00e+00 (allowed 1e-12)
      Source: R vegan::estimateR (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/estimateR.html (version 2.7.8)
agrees  BCI plot 1 | add_diversity_columns (hill numbers) | hill number 1: simplydata 55.6127038814, R vegan::renyi 55.6127038814; relative difference 0.00e+00 (allowed 1e-12)
      Source: R vegan::renyi (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/renyi.html (version 2.7.8)
agrees  BCI plot 1 | add_diversity_columns (hill numbers) | hill number 1: simplyd

## Bray-Curtis distances between the 20 dune meadows: `find_distances_between_rows`

All 190 distances against SciPy `pdist(metric="braycurtis")` and vegan `vegdist`, compared by the largest
relative difference over the 190 pairs (which must be at most 1e-12), and three single distances.

In [4]:
found = rows(load("dune").find_distances_between_rows, columns=dune_species)
distance = [[row[str(j + 1)] for j in range(len(found))] for row in found]
pairs = [distance[i][j] for i in range(len(found)) for j in range(i + 1, len(found))]
checks.check("dune", "find_distances_between_rows", "distance of each pair", pairs)
checks.check_values("dune", "find_distances_between_rows",
                    {f"meadow {i + 1} to {j + 1}": distance[i][j] for i, j in ((0, 1), (0, 19), (7, 15))})

agrees  dune | find_distances_between_rows | distance of each pair: largest relative difference over 190 values: simplydata 0, SciPy pdist (braycurtis) 0; difference 0.00e+00 (allowed 1e-12)
      Source: SciPy pdist (braycurtis) (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.pdist.html
agrees  dune | find_distances_between_rows | distance of each pair: largest relative difference over 190 values: simplydata 0, R vegan::vegdist 0; difference 0.00e+00 (allowed 1e-12)
      Source: R vegan::vegdist (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/vegdist.html (version 2.7.8)
agrees  dune | find_distances_between_rows | meadow 1 to 2: simplydata 0.466666666667, SciPy pdist (braycurtis) 0.466666666667; relative difference 1.19e-16 (allowed 1e-12)
      Source: SciPy pdist (braycurtis) (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.pdist.html
agrees  dune | find_distances_between_rows

3

## PERMANOVA and the spread check: `compare_profiles_of`

Do the four kinds of management (biological, hobby, nature conservation, standard farming) differ in the make-up
of the dune meadows? PERMANOVA on Bray-Curtis (R `vegan::adonis2`), then the spread check (`betadisper` with
`permutest`). Pseudo-F and R squared are also computed from the distances by Anderson's (2001) formula:
SS_total = sum of d² / N over all pairs, SS_within = the same within each group divided by its size,
F = ((SS_total - SS_within) / (a - 1)) / (SS_within / (N - a)). The second dataset is mite by microtopography
(Blanket or Hummock), also on Bray-Curtis. `method="anosim"`: ANOSIM's R (Clarke 1993) is
(mean rank between - mean rank within) / (M / 2) over the M ranked distances (R `vegan::anosim`).

In [5]:
load("dune").compare_profiles_of(dune_species, for_each_group_in="Management")

The groups of "Management" (BF, HF, NM, SF) explain 34.2 % of the differences between the rows (R squared = 0.342; Bray-Curtis distances over 30 columns (Achimill, Agrostol, Airaprae, Alopgeni, Anthodor, Bellpere and 24 more); pseudo-F = 2.77, from 999 permutations).
If the groups truly had the same make-up, a difference in make-up this big or bigger would turn up in about 2 of 1000 studies like this one (p = 0.002).
Test: PERMANOVA (permutational analysis of variance on distances), run in R: set.seed(1); vegan::adonis2(distances ~ Management, permutations = 999) with distances = vegan::vegdist(x, "bray").
Spread check: the average distance to the group centre is 0.220 in BF, 0.279 in HF, 0.451 in NM and 0.364 in SF (as vegan::betadisper and permutest: F = 1.95, p = 0.159).
Note: the rows have very different totals (from 15 to 48); Bray-Curtis on raw counts mixes those totals (such as sequencing depth) with the make-up. Turn the counts into shares first, or use measure="aitchison".


,test,what it compares,distance,statistic name,statistic,degrees of freedom 1,degrees of freedom 2,R squared,p-value,permutations,spread check F,spread check p-value,count in BF,distance to centre in BF,count in HF,distance to centre in HF,count in NM,distance to centre in NM,count in SF,distance to centre in SF
1,PERMANOVA,"the make-up of 30 columns (Achimill, Agrostol,...",Bray-Curtis,pseudo-F,2.767243,3.0,16.0,0.341611,0.002,999,1.95064,0.159,3,0.219582,5,0.279148,6,0.450971,6,0.363963


In [6]:
for name, group, columns in (("dune", "Management", dune_species), ("mite", "Topo", mite_species)):
    found = rows(load(name).compare_profiles_of, columns, for_each_group_in=group)[0]
    checks.check_values(name, "compare_profiles_of (PERMANOVA)",
                        {"pseudo-F": found["statistic"], "R squared": found["R squared"],
                         "p-value (seed 1)": found["p-value"], "spread check F": found["spread check F"],
                         "spread check p-value (seed 1)": found["spread check p-value"]})
    found = rows(load(name).compare_profiles_of, columns, for_each_group_in=group, method="anosim")[0]
    checks.check_values(name, "compare_profiles_of (ANOSIM)",
                        {"ANOSIM R": found["statistic"], "p-value (seed 1)": found["p-value"]})

agrees  dune | compare_profiles_of (PERMANOVA) | pseudo-F: simplydata 2.76724349818, Anderson (2001) formula 2.76724349818; relative difference 1.60e-16 (allowed 1e-10)
      Source: Anderson (2001) formula (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/11_communities.py
agrees  dune | compare_profiles_of (PERMANOVA) | pseudo-F: simplydata 2.76724349818, R vegan::adonis2 2.76724349818; relative difference 0.00e+00 (allowed 1e-10)
      Source: R vegan::adonis2 (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/adonis2.html (version 2.7.8)
agrees  dune | compare_profiles_of (PERMANOVA) | R squared: simplydata 0.341610672393, Anderson (2001) formula 0.341610672393; relative difference 0.00e+00 (allowed 1e-10)
      Source: Anderson (2001) formula (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/11_communities.py
agrees  dune | compare_profiles_of (PERMANOVA) | R squared: 

agrees  dune | compare_profiles_of (ANOSIM) | ANOSIM R: simplydata 0.257870590097, Clarke (1993) formula 0.257870590097; relative difference 0.00e+00 (allowed 1e-12)
      Source: Clarke (1993) formula (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/11_communities.py
agrees  dune | compare_profiles_of (ANOSIM) | ANOSIM R: simplydata 0.257870590097, R vegan::anosim 0.257870590097; relative difference 0.00e+00 (allowed 1e-12)
      Source: R vegan::anosim (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/anosim.html (version 2.7.8)
agrees  dune | compare_profiles_of (ANOSIM) | p-value (seed 1): simplydata 0.011, R vegan::anosim 0.011; difference 9.89e-17 (allowed 1e-12)
      Source: R vegan::anosim (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/anosim.html (version 2.7.8)
agrees  mite | compare_profiles_of (PERMANOVA) | pseudo-F: simplydata 12.7056046584, Anderson (2001) formula 12.7056046584; 

agrees  mite | compare_profiles_of (ANOSIM) | ANOSIM R: simplydata 0.253112053171, Clarke (1993) formula 0.253112053171; relative difference 0.00e+00 (allowed 1e-12)
      Source: Clarke (1993) formula (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/11_communities.py
agrees  mite | compare_profiles_of (ANOSIM) | ANOSIM R: simplydata 0.253112053171, R vegan::anosim 0.253112053171; relative difference 0.00e+00 (allowed 1e-12)
      Source: R vegan::anosim (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/anosim.html (version 2.7.8)
agrees  mite | compare_profiles_of (ANOSIM) | p-value (seed 1): simplydata 0.001, R vegan::anosim 0.001; difference 0.00e+00 (allowed 1e-12)
      Source: R vegan::anosim (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/anosim.html (version 2.7.8)


## Maps of the rows: principal coordinates, NMDS and correspondence analysis

- `add_principal_coordinates` (PCoA, Gower 1966) of the dune Bray-Curtis distances: the eigenvectors of the
  double-centred matrix -½ d², scaled by the square root of their eigenvalues (NumPy `eigh`), and R
  `stats::cmdscale`.
- `add_nmds` (R `vegan::metaMDS`, no automatic transformation, seed 1): Kruskal's stress-1 against metaMDS run the
  same way (read with `show="numbers"`), and the scores by their absolute value.
- `add_correspondence_axes` (CA) of varespec: Oksanen's vegan tutorial prints the eigenvalues 0.5249 and 0.3568
  of a total inertia 2.083, so CA1 holds 25.2 % and CA2 17.1 %: the shares `show="numbers"` gives back, against
  the printed shares (half a unit of the last digit) and the singular values of the matrix of chi-square
  residuals (NumPy `svd`, 1e-10); the site scores against vegan `cca` (scaling "sites"). The eigenvalues and the
  total inertia, also given back by `show="numbers"` (simplydata a5a8ff0 and later), against the printed values.

In [7]:
found = rows(load("dune").add_principal_coordinates, columns=dune_species)
for k in (1, 2):
    checks.check_values("dune", "add_principal_coordinates",
                        {f"|PCo{k}| meadow {i + 1}": abs(found[i][f"PCo{k}"]) for i in (0, 9, 19)})

for name, columns in (("dune", dune_species), ("varespec", vare_species)):
    found = rows(load(name).add_nmds, columns=columns)
    checks.check_values(name, "add_nmds", {f"|NMDS{k}| row 1": abs(found[0][f"NMDS{k}"]) for k in (1, 2)})
    stress = read_row(quiet(load(name).add_nmds, columns=columns, show="numbers"))["stress"]
    checks.check(name, "add_nmds", "stress (printed, 3 decimals)", stress)

found = rows(load("varespec").add_correspondence_axes, columns=vare_species)
for k in (1, 2):
    checks.check_values("varespec", "add_correspondence_axes",
                        {f"|CA{k}| site {i + 1}": abs(found[i][f"CA{k}"]) for i in (0, 11, 23)})
shares = read_row(quiet(load("varespec").add_correspondence_axes, columns=vare_species, show="numbers"))
for k in (1, 2):
    percent = 100 * shares[f"CA{k} share of the inertia"]
    checks.check("varespec", "add_correspondence_axes", f"CA{k} % (printed)", percent)
    checks.check("varespec", "add_correspondence_axes", f"CA{k} %", percent)
for quantity in ("CA1 eigenvalue", "CA2 eigenvalue", "total inertia"):
    checks.check("varespec", "add_correspondence_axes", quantity, shares[quantity])

agrees  dune | add_principal_coordinates | |PCo1| meadow 1: simplydata 0.354731824459, Gower (1966), NumPy eigh 0.354731824459; relative difference 1.25e-15 (allowed 1e-10)
      Source: Gower (1966), NumPy eigh (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/11_communities.py
agrees  dune | add_principal_coordinates | |PCo1| meadow 1: simplydata 0.354731824459, R stats::cmdscale 0.354731824459; relative difference 0.00e+00 (allowed 1e-10)
      Source: R stats::cmdscale (run reference) - https://search.r-project.org/R/refmans/stats/html/cmdscale.html
agrees  dune | add_principal_coordinates | |PCo1| meadow 10: simplydata 0.311237265883, Gower (1966), NumPy eigh 0.311237265883; relative difference 1.78e-16 (allowed 1e-10)
      Source: Gower (1966), NumPy eigh (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/11_communities.py
agrees  dune | add_principal_coordinates | |PCo1| meadow 10: simp

agrees  varespec | add_nmds | stress (printed, 3 decimals): simplydata 0.100021070584, R vegan::metaMDS (seed 1) 0.1; difference 2.11e-05 (allowed 5e-04)
      Source: R vegan::metaMDS (seed 1) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/metaMDS.html (version 2.7.8)


agrees  varespec | add_correspondence_axes | |CA1| site 1: simplydata 0.108121658834, R vegan::cca (scaling sites) 0.108121658834; relative difference 7.70e-16 (allowed 1e-10)
      Source: R vegan::cca (scaling sites) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/cca.html (version 2.7.8)
agrees  varespec | add_correspondence_axes | |CA1| site 12: simplydata 0.38286558433, R vegan::cca (scaling sites) 0.38286558433; relative difference 1.45e-16 (allowed 1e-10)
      Source: R vegan::cca (scaling sites) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/cca.html (version 2.7.8)
agrees  varespec | add_correspondence_axes | |CA1| site 24: simplydata 0.259526631817, R vegan::cca (scaling sites) 0.259526631817; relative difference 2.14e-16 (allowed 1e-10)
      Source: R vegan::cca (scaling sites) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/cca.html (version 2.7.8)
agrees  varespec | add_correspondence_axes | |CA2| site

## Profiles explained by other columns, and profiles against profiles

- `explain_profiles_of`: distance-based RDA of varespec on Bray-Curtis with N, P and K (R `vegan::dbrda`;
  R squared and adjusted R squared by `RsquareAdj`, marginal tests by `anova(by = "margin")` after `set.seed(1)`),
  and the canonical correspondence analysis of Oksanen's tutorial, `cca(varespec ~ Al + P + K, varechem)`.
- `find_correlation_between_profiles`: the Mantel test of varespec (Bray-Curtis) against the 14 soil variables
  (Euclidean distances of the standardised columns), R `vegan::mantel` after `set.seed(1)`; the statistic is also
  the Pearson correlation of the two sets of 276 distances (SciPy).

In [8]:
model = {row["term"]: row for row in rows(load("varespec").explain_profiles_of, vare_species, using=["N", "P", "K"])}
values = {"R squared": model["all together"]["R squared"],
          "adjusted R squared": model["all together"]["adjusted R squared"]}
for term in ("N", "P", "K"):
    values[f"pseudo-F of {term} (marginal)"] = model[term]["pseudo-F"]
    values[f"p-value of {term} (seed 1)"] = model[term]["p-value"]
checks.check_values("varespec", "explain_profiles_of (db-RDA)", values)

model = {row["term"]: row for row in rows(load("varespec").explain_profiles_of, vare_species, using=["Al", "P", "K"],
                                          method="canonical correspondence")}
checks.check_values("varespec", "explain_profiles_of (CCA)", {"R squared": model["all together"]["R squared"],
                                                              "pseudo-F": model["all together"]["pseudo-F"]})

found = rows(load("varespec").find_correlation_between_profiles, first=vare_species, second=VARECHEM)[0]
checks.check_values("varespec and varechem", "find_correlation_between_profiles (Mantel)",
                    {"Mantel r": found["statistic"], "p-value (seed 1)": found["p-value"]})

agrees  varespec | explain_profiles_of (db-RDA) | R squared: simplydata 0.195913058887, R vegan::dbrda 0.195913058887; relative difference 0.00e+00 (allowed 1e-10)
      Source: R vegan::dbrda (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/dbrda.html (version 2.7.8)
agrees  varespec | explain_profiles_of (db-RDA) | adjusted R squared: simplydata 0.0753000177198, R vegan::dbrda 0.0753000177198; relative difference 5.53e-16 (allowed 1e-10)
      Source: R vegan::dbrda (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/dbrda.html (version 2.7.8)
agrees  varespec | explain_profiles_of (db-RDA) | pseudo-F of N (marginal): simplydata 1.53157846889, R vegan::dbrda anova(by = 'margin') 1.53157846889; relative difference 1.45e-16 (allowed 1e-10)
      Source: R vegan::dbrda anova(by = 'margin') (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/dbrda.html (version 2.7.8)
agrees  varespec | explain_profiles_of (db-RDA) | p-value of 

agrees  varespec | explain_profiles_of (CCA) | R squared: simplydata 0.309204040715, R vegan::cca 0.309204040715; relative difference 1.80e-16 (allowed 1e-10)
      Source: R vegan::cca (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/cca.html (version 2.7.8)
agrees  varespec | explain_profiles_of (CCA) | pseudo-F: simplydata 2.98403637677, R vegan::cca 2.98403637677; relative difference 0.00e+00 (allowed 1e-10)
      Source: R vegan::cca (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/cca.html (version 2.7.8)


agrees  varespec and varechem | find_correlation_between_profiles (Mantel) | Mantel r: simplydata 0.304745412698, Pearson r of the distances (SciPy) 0.304745412698; relative difference 1.82e-16 (allowed 1e-12)
      Source: Pearson r of the distances (SciPy) (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/11_communities.py
agrees  varespec and varechem | find_correlation_between_profiles (Mantel) | Mantel r: simplydata 0.304745412698, R vegan::mantel 0.304745412698; relative difference 1.82e-16 (allowed 1e-12)
      Source: R vegan::mantel (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/mantel.html (version 2.7.8)
agrees  varespec and varechem | find_correlation_between_profiles (Mantel) | p-value (seed 1): simplydata 0.001, R vegan::mantel 0.001; difference 0.00e+00 (allowed 1e-12)
      Source: R vegan::mantel (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/mantel.html (version 2.7.8)


3

## Clusters of meadows, and log ratios of mite counts

- `add_clusters(method="hierarchical", measure="bray-curtis", how_many=3)`: average linkage (UPGMA) on Bray-Curtis.
  The join heights of the drawn tree (read with `chart.get_numbers_drawn()`: each join is one drawn line, whose
  highest point is the height of the join) against SciPy `linkage(method="average")` (the parent) and R
  `hclust`; the clusters against R `cutree` (numbered in the order
  they first appear); the silhouette widths
  against R `cluster::silhouette`.
- `add_log_ratio_columns`: centred log ratios (clr) of the 35 mite species. With `zeros_replaced_by="adding 1"`
  the clr is log(x + 1) minus the row mean of log(x + 1) (NumPy; vegan `decostand(method = "clr",
  pseudocount = 1)`). By default zeros are first replaced by R `zCompositions::cmultRepl(method = "GBM")`, an
  iterative estimate, then `compositions::clr`.

In [9]:
found = rows(load("dune").add_clusters, columns=dune_species, how_many=3, method="hierarchical", measure="bray-curtis",
             show="silhouette")
widths = [row["silhouette width"] for row in found]
checks.check("dune", "add_clusters (hierarchical, 3)", "average silhouette width", sum(widths) / len(widths))
checks.check_values("dune", "add_clusters (hierarchical, 3)",
                    {f"silhouette width of meadow {i + 1}": widths[i] for i in (0, 9, 19)})
first_seen = {}
numbered = [first_seen.setdefault(row["cluster"], len(first_seen) + 1) for row in found]
checks.check("dune", "add_clusters (hierarchical, 3)", "cluster of each meadow (numbered in order of first appearance)",
             numbered)
chart = load("dune").start_chart()
quiet(chart.draw_cluster_tree, columns=dune_species, measure="bray-curtis")
joins = {}
for point in read_rows(quiet(chart.get_numbers_drawn)):
    joins[point["layer"]] = max(joins.get(point["layer"], 0.0), point["y"])
checks.check("dune", "draw_cluster_tree (average, Bray-Curtis)", "top join", max(joins.values()))
checks.check("dune", "draw_cluster_tree (average, Bray-Curtis)", "lowest join", min(joins.values()))

found = rows(load("mite").add_log_ratio_columns, columns=mite_species, zeros_replaced_by="adding 1")
checks.check("mite", "add_log_ratio_columns (adding 1)", "clr of each value (row by row)",
             [row["clr_" + s] for row in found for s in mite_species])
found = rows(load("mite").add_log_ratio_columns, columns=mite_species)
checks.check_values("mite", "add_log_ratio_columns (estimated small values)",
                    {f"clr row {i + 1}, {mite_species[j]}": found[i]["clr_" + mite_species[j]]
                     for i, j in ((0, 0), (1, 34), (69, 0), (35, 17))})

agrees  dune | add_clusters (hierarchical, 3) | average silhouette width: simplydata 0.285531716383, R cluster::silhouette 0.285531716383; relative difference 3.89e-16 (allowed 1e-12)
      Source: R cluster::silhouette (run reference) - https://search.r-project.org/CRAN/refmans/cluster/html/silhouette.html (version 2.1.8.3)
agrees  dune | add_clusters (hierarchical, 3) | silhouette width of meadow 1: simplydata 0.328489278246, R cluster::silhouette 0.328489278246; relative difference 3.38e-16 (allowed 1e-12)
      Source: R cluster::silhouette (run reference) - https://search.r-project.org/CRAN/refmans/cluster/html/silhouette.html (version 2.1.8.3)
agrees  dune | add_clusters (hierarchical, 3) | silhouette width of meadow 10: simplydata 0.272769371758, R cluster::silhouette 0.272769371758; relative difference 4.07e-16 (allowed 1e-12)
      Source: R cluster::silhouette (run reference) - https://search.r-project.org/CRAN/refmans/cluster/html/silhouette.html (version 2.1.8.3)
agrees  du

agrees  mite | add_log_ratio_columns (adding 1) | clr of each value (row by row): largest relative difference over 2450 values: simplydata 1.66247268494e-14, formula log(x + 1) - row mean 0; difference 1.66e-14 (allowed 1e-12)
      Source: formula log(x + 1) - row mean (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/11_communities.py
agrees  mite | add_log_ratio_columns (adding 1) | clr of each value (row by row): largest relative difference over 2450 values: simplydata 0, R vegan::decostand(clr, pseudocount = 1) 0; difference 0.00e+00 (allowed 1e-12)
      Source: R vegan::decostand(clr, pseudocount = 1) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/decostand.html (version 2.7.8)


agrees  mite | add_log_ratio_columns (estimated small values) | clr row 1, Brachy: simplydata 2.65706131913, R zCompositions::cmultRepl (GBM) + compositions::clr 2.65706131913; relative difference 0.00e+00 (allowed 1e-06)
      Source: R zCompositions::cmultRepl (GBM) + compositions::clr (run reference) - https://search.r-project.org/CRAN/refmans/zCompositions/html/cmultRepl.html (version 1.6.2)
agrees  mite | add_log_ratio_columns (estimated small values) | clr row 2, Trimalc2: simplydata -0.557538098366, R zCompositions::cmultRepl (GBM) + compositions::clr -0.557538098366; relative difference 1.99e-16 (allowed 1e-06)
      Source: R zCompositions::cmultRepl (GBM) + compositions::clr (run reference) - https://search.r-project.org/CRAN/refmans/zCompositions/html/cmultRepl.html (version 1.6.2)
agrees  mite | add_log_ratio_columns (estimated small values) | clr row 70, Brachy: simplydata 1.74464633225, R zCompositions::cmultRepl (GBM) + compositions::clr 1.74464633225; relative differenc

4

## Result

In [10]:
results = checks.finish()

11_communities: 792 comparisons - 792 agrees
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)
